In [ ]:
import marimo as mo

# Task 5 — Judge Model

We build an automated judge: an LLM that grades descriptions using the same
rubric defined in Task 1.

## Model Choice

- **Judge model**: `google/gemma-2-9b-it` (model NOT used for generation)
- If Gemma struggles with structured output, we fall back to `Qwen/Qwen3-30B-A3B-Instruct-2507`
- **Temperature**: 0.0 — deterministic judging for reproducibility

## Why `explanation` Before `verdict` in the Schema

LLMs generate tokens left-to-right. If `verdict` came first, the model would build all the answer around that score - a form of priming/anchoring bias.

We're doing it the other way: first prompting for explanation, then summarizing that information to verdict.
This is the LLM equivalent of chain-of-thought prompting embedded in the schema.

## Why use Pydantic at all?

Pydantic gives us a strict contract for judge output: every criterion must be present,
every verdict must be one of `good/ok/bad`. We also use tenacity to retry on fails.

In [ ]:
from _bootstrap import bootstrap_notebook
bootstrap_notebook()

import pandas as pd
from tqdm import tqdm

from src.artifacts import load_csv_artifact
from src.config import PROMPT_JUDGE_ALL, get_force_rerun, get_judge_config, prompt_path
from src.judge_runtime import build_all_criteria_prompt, create_all_at_once_judge
from src.paths import ASSIGNMENT_XLSX_PATH, TASK_05_JUDGE_SANITY_CSV_PATH
from src.runtime import load_project_env, read_text, setup_mlflow
from src.utils import format_judge_input

load_project_env()
setup_mlflow("judge_runs")

PosixPath('/Users/zakhar/Documents/code/learning-ai-nebius-ai-engineering-2026/w2-ai-product/outputs/experiments.db')

In [ ]:
FORCE_RERUN = get_force_rerun()
judge_config = get_judge_config()
JUDGE_MODEL = judge_config.model
prompt_template = read_text(prompt_path(PROMPT_JUDGE_ALL))
JUDGE_PROMPT = build_all_criteria_prompt(prompt_template)
print(f"Judge model: {JUDGE_MODEL}")
if FORCE_RERUN:
    print("FORCE_RERUN=1 — ignoring existing Task 5 sanity artifact.")
print("\n--- Judge Prompt ---")
print(JUDGE_PROMPT)

Judge model: nebius/google/gemma-2-9b-it-fast

--- Judge Prompt ---
You are an expert product description evaluator. Your task is to rate a generated product description against an evaluation rubric.

You will receive:
1. The ORIGINAL PRODUCT DATA (name, attributes, material, warranty)
2. The GENERATED DESCRIPTION to evaluate

For each criterion below, you must:
- First write your reasoning (explanation)
- Then give a verdict: "good", "ok", or "bad"

## Evaluation Rubric

### Fluency — Natural, easy-to-read sentences
- good: Natural, easy-to-read flow; no awkward phrasing or unnatural constructions
- ok:   Mostly natural but 1–2 awkward phrases or choppy transitions
- bad:  Stilted, robotic, or hard to follow; multiple unnatural constructions

### Grammar — Correct spelling and punctuation
- good: Zero spelling or punctuation errors
- ok:   1–2 minor errors that don't impede understanding
- bad:  3+ errors, or any error that changes meaning or is immediately visible

### Tone — Matches

## Judge Function with Retry Logic

In [ ]:
run_judge = create_all_at_once_judge(
    model=JUDGE_MODEL,
    prompt=JUDGE_PROMPT,
    format_judge_input=format_judge_input,
)

## Sanity Check — 5 Products

Run the judge on 5 products and review manually before the full run.
Check: does the judge apply the rubric correctly? Are explanations coherent?

In [ ]:
required_columns = [
    "product_name",
    "description",
    "judge_status",
    "judge_error",
]
existing_df = None
if not FORCE_RERUN:
    existing_df = load_csv_artifact(
        TASK_05_JUDGE_SANITY_CSV_PATH,
        required_columns=required_columns,
    )

if existing_df is not None:
    print(f"Source: loaded existing artifact {TASK_05_JUDGE_SANITY_CSV_PATH}")
    sanity_results = existing_df.to_dict("records")
else:
    print(
        f"Source: artifact missing or invalid, running live sanity check -> {TASK_05_JUDGE_SANITY_CSV_PATH}"
    )
    df_main = pd.read_excel(ASSIGNMENT_XLSX_PATH)
    sanity_sample = df_main.head(5)

    sanity_results = []
    for _, row in tqdm(sanity_sample.iterrows(), total=5, desc="Sanity check"):
        try:
            result = run_judge(row.to_dict(), str(row["generated_description"]))
            ratings = result.to_ratings()
            sanity_results.append({
                "product_name": row["product_name"],
                "description": row["generated_description"],
                "judge_status": "ok",
                "judge_error": "",
                **{f"{k}_verdict": v for k, v in ratings.items()},
                **{f"{k}_explanation": getattr(result, k).explanation for k in ratings},
            })
        except Exception as e:
            print(f"Failed on {row['product_name']}: {e}")
            sanity_results.append({
                "product_name": row["product_name"],
                "description": row["generated_description"],
                "judge_status": "error",
                "judge_error": str(e),
            })

    pd.DataFrame(sanity_results).to_csv(TASK_05_JUDGE_SANITY_CSV_PATH, index=False)
    print(f"Saved sanity results to {TASK_05_JUDGE_SANITY_CSV_PATH}")

Source: loaded existing artifact /Users/zakhar/Documents/code/learning-ai-nebius-ai-engineering-2026/w2-ai-product/outputs/task_05_judge_sanity.csv


In [ ]:
# Display each sanity result with explanations
_md = "## Sanity Check Results\n\n"
for _r in sanity_results:
    if _r.get("judge_status") != "ok":
        _md += f"**{_r['product_name']}**: ERROR — {_r['judge_error']}\n\n"
    else:
        _md += f"### {_r['product_name']}\n"
        for _c in ["fluency", "grammar", "tone", "length", "grounding"]:
            _md += f"- **{_c}** [{_r[f'{_c}_verdict']}]: {_r[f'{_c}_explanation']}\n"
        _md += "\n"
mo.md(_md)

## Sanity Check Results

### Apple iPhone 15 Pro
- **fluency** [good]: The sentences flow well and are easy to read. There are no awkward phrases or unnatural constructions.
- **grammar** [good]: There are no spelling or punctuation errors.
- **tone** [good]: The tone is friendly, enthusiastic, and persuasive without being overly hyperbolic. It effectively highlights the phone's features and benefits.
- **length** [good]: The description is 89 words long, which falls within the acceptable range of 50-90 words.
- **grounding** [good]: All claims in the description are directly traceable to the provided product data. There are no invented features or specifications.

### Samsung Galaxy S24 Ultra
- **fluency** [good]: The sentences flow naturally and are easy to read. There are no awkward phrases or unnatural constructions.
- **grammar** [good]: There are no spelling or punctuation errors.
- **tone** [good]: The tone is friendly, credible, and persuasive without being pushy or hyperbolic. It effectively highlights the phone's key features and benefits.
- **length** [good]: The description is 87 words long, which falls within the acceptable range of 50-90 words.
- **grounding** [good]: All claims in the description can be traced back to the original product data. For example, the 200MP camera, S-Pen support, 120Hz AMOLED display, Armor Aluminum frame, Gorilla Glass Victus, sustainable sourcing, and 1-year warranty are all mentioned in the data.

### Google Pixel 8 Pro
- **fluency** [good]: The sentences flow well and are easy to read. There are no awkward phrases or unnatural constructions.
- **grammar** [good]: There are no spelling or punctuation errors.
- **tone** [good]: The tone is friendly, credible, and persuasive without being pushy. It uses language that appeals to potential customers.
- **length** [ok]: The description is 84 words long, which falls within the acceptable range of 50-90 words.
- **grounding** [good]: All claims in the description are traceable to the original product data. For example, the Tensor G3 chip, Magic Eraser, 50MP camera, matte glass back, aluminum frame, 4.7/5 rating, and 1-year limited warranty are all mentioned in the data.

### Sony WH‑1000XM5 Headphones
- **fluency** [good]: The sentences flow well and are easy to read. There are no awkward phrases or unnatural constructions.
- **grammar** [good]: There are no spelling or punctuation errors.
- **tone** [good]: The tone is friendly, credible, and persuasive without being pushy. It uses language that appeals to the customer's desire for a good listening experience.
- **length** [good]: The description is 88 words long, which falls within the acceptable range of 50-90 words.
- **grounding** [good]: All claims in the description can be traced back to the original product data. For example, 'advanced active noise cancelling technology' is mentioned as a feature, 'up to 30 hours of uninterrupted listening' corresponds to the battery life, and 'Bluetooth 5.2 connectivity' is listed as an attribute. The description also accurately reflects the material and warranty information.

### Bose QuietComfort Ultra Earbuds
- **fluency** [good]: The sentences flow well and are easy to read. There are no awkward phrases or unnatural constructions.
- **grammar** [good]: There are no spelling or punctuation errors.
- **tone** [good]: The tone is friendly and enthusiastic, using persuasive language like 'immerse yourself' and 'crystal-clear quality' without being overly pushy.
- **length** [good]: The description is 70 words long, which falls within the 50-90 word target range.
- **grounding** [good]: All claims are directly supported by the original product data. For example, 'CustomTune sound calibration', 'Advanced Noise Cancellation (ANC)', 'IPX4 rating', and 'silicone ear tips' are all mentioned in the data.